### Cell 1: Install dependencies

In [ ]:
!pip install -q sentence-transformers bitsandbytes accelerate

### Cell 2: Imports and helpers

In [ ]:
import gc
import time
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import BitsAndBytesConfig

MODEL_NAME = "google/embeddinggemma-300m"

# Store results
results = {}

def get_vram_mb() -> float:
    """Current GPU VRAM allocation in MB."""
    return torch.cuda.memory_allocated(0) / (1024 * 1024)

def cleanup():
    """Free up GPU memory."""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

def benchmark(model, sentences, n_runs=10):
   """
   Generate embeddings, measure VRAM and inference time.
   
   Returns a dict with vram_mb, embeddings, mean_time_ms, std_time_ms.
   """
   vram = get_vram_mb()

   # Warmup - run once to load model and stabilize performance
   _ = model.encode(sentences[:2], show_progress_bar=False)

   # Timed runs
   times = []
   for _ in range(n_runs):
        start = time.perf_counter()
        embeddings = model.encode(
           sentences, batch_size=32, show_progress_bar=False, convert_to_numpy=True
        )
        torch.cuda.synchronize() # Ensure all GPU operations are complete before stopping timer
        times.append(time.perf_counter() - start)

    return {
       "vram_mb": vram,
       "embeddings": embeddings,
       "mean_time_ms": np.mean(times) * 1000,
       "std_time_ms": np.std(times) * 1000
    }

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

### Cell 3: Test sentences

In this case, these are a mix of real excerpts from SEC filings and short search queries.

In [ ]:
# Want to test that quantisation preserves semantic nuance across different types of content.
test_sentences = [
    # across different content types and lengths from real SEC filings
    "•developments or disputes concerning our intellectual property or other proprietary rights;"
    "We are subject to claims, lawsuits, regulatory and government inquiries and investigations, other proceedings, and orders involving competition, intellectual property, data privacy and security, tax and related compliance, labor and employment, commercial disputes, content generated by our users, goods and services offered by advertisers or publishers using our platforms, personal injury, and other matters."
    "We are exposed to financial market risks, including changes in foreign currency exchange rates, interest rates, and equity investment risks."
    "Our marketable and non-marketable equity securities are subject to a wide variety of market-related risks that could substantially reduce or increase the fair value of our holdings."
    "Innovations in our products and services could also result in changes to user and customer behavior and affect our revenue trends. These endeavors involve significant risks and uncertainties, including diversion of resources and management attention from current operations, different monetization models, and the use of alternative investment, governance, or compensation structures that may fail to adequately align incentives across the company or otherwise accomplish their objectives."
    "From time to time, we acquire or invest in businesses, content, and technologies that support our business. The risks associated with such acquisitions or investments (some of which may be unforeseen) include the difficulty of integrating solutions, operations, and personnel; inheriting liabilities and exposure to litigation; failure to realize anticipated benefits and expected synergies; and diversion of management’s time and attention, among other acquisition-related risks."
    "We view our employees and our culture as key to our success.  As of December 31, 2024, we had approximately 14,000 full-time employees.  Of these, approximately 9,600 (69%) were located in the United States and Canada, 2,200 (16%) in Europe, Middle East, and Africa, 600 (4%) in Latin America and 1,600 (11%) in Asia-Pacific.  We also have a number of employees engaged in content production, some of whom are part-time or temporary, and whose numbers fluctuate throughout the year and may be covered by collective bargaining agreements"
    "As of December 27, 2025, we had approximately 31,000 employees in our global workforce. We believe we are at our best when our culture of innovation and people from all kinds of backgrounds work together in an engaging and open environment. Areas of focus for us include:"
    "Volatility in our stock price could adversely affect our business and financing opportunities and force us to increase our cash compensation to employees or grant larger stock awards than we have historically, which could hurt our operating results or reduce the percentage ownership of our existing stockholders, or both."
    "The semiconductor industry is highly cyclical and has experienced severe downturns."
    "We believe trade dynamics and geopolitics are disrupting and reshaping global supply chains and affecting customer order behavior. Our global manufacturing capabilities enable us to support our customers’ needs. The overall semiconductor market recovery is continuing, though at a slower pace than prior upturns, likely related to broader macroeconomic dynamics and overall uncertainty."
    "The “semiconductor cycle” refers to the ebb and flow of supply and demand and the building and depleting of inventories. The semiconductor market historically has been characterized by periods of tight supply caused by strengthening demand and/or insufficient manufacturing capacity, followed by periods of surplus inventory caused by weakening demand and/or excess manufacturing capacity. These are typically referred to as upturns and downturns in the semiconductor cycle. Semiconductor cycles are affected by the significant time and money required to build and maintain semiconductor manufacturing facilities."
    "The terms of certain of our debt facilities contain, and any of our other future debt agreements may contain, covenant restrictions that may limit our ability to operate our business, including restrictions on our and/or our subsidiaries’ ability to, among other things, incur additional debt or create liens. In addition, under certain circumstances we are required to maintain a certain amount of liquidity. As a result of these covenants, our ability to respond to changes in business and economic conditions and engage in beneficial transactions, including to obtain additional financing as needed, may be restricted. Furthermore, our failure to comply with our debt covenants could result in a default under our debt agreements, which could permit the holders to accelerate our obligation to repay the debt. If any of our debt is accelerated, we may not have sufficient funds available to repay it."
    "We have financial instruments that are subject to interest rate risk, principally fixed-rate debt obligations. The investors in our fixed-rate debt obligations do not generally have the right to demand we pay off these obligations prior to maturity. Therefore, exposure to interest rate risk is not believed to be material for our fixed-rate debt."
    "•changes to U.S. and non-U.S. government policies, including sourcing restrictions, requirements to expend a portion of program funds locally and governmental industrial cooperation or participation requirements;"
    "•sanctions, import and export controls, other market access barriers, political unrest, geopolitical tensions, changes in regimes, or armed conflict (such as ongoing conflicts in the Middle East and Ukraine), any of which may affect our business continuity, increase our operating costs, limit demand for our products and services, limit our ability to source components or final products, or prevent or impede us from operating in certain jurisdictions, complying with local laws, or offering products or services;"
    "•an evolving foreign policy landscape that could harm our revenues and could subject us to litigation, new regulatory costs and challenges (including new customer requirements), uncertainty regarding regulatory outcomes, and other liabilities under local laws that may not offer due process or clear legal precedent;"
    "As a corporation with an international presence, we are subject to risks and uncertainties caused by significant events with macroeconomic impacts, including, but not limited to, geopolitical tensions, heightened interest rates, monetary policy changes,  foreign currency fluctuations, and the imposition of tariffs or other impacts on trade relations. Additionally, these macroeconomic impacts have disrupted, and may continue to disrupt, the operations of our customers and prospective customers. We continuously monitor the direct and indirect impacts of these circumstances on our business and financial results, as well as the overall global economy and geopolitical landscape."
    "•changes to the content or application of third-party policies that limit our ability to deliver, target, or measure the effectiveness of advertising, including changes by mobile operating system and browser providers such as Apple and Google;"
    # Short search queries
    "What are the legal issues?"
    "What are the risks?"
    "What are the geopolitical risks for company?"
    "How many employees does the company have?"
    "Executive compensation and stock options"
    "Semiconductor supply chain dependency"
    "Debt covenant compliance risk"
    "revenue recognition policy changes"
]

print(f"{len(test_sentences)} test sentences ready")

### Cell 4: FP32 Baseline